# EDA: Feature Attribution Rules for LLM Analyst

**Goal:** Find the strongest drivers of salary deviations from per-category baselines.  
These rules will feed dynamic context into the LLM prompts so narratives cite the *actual* primary driver.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Load raw dataset
df = pd.read_csv("data/ds_salaries.csv")

# Load feature mappings to engineer the same features the model uses
mappings = joblib.load("model/feature_mappings.joblib")
job_map = mappings["job_category_map"]
tier_map = mappings["country_tier_map"]

# Engineer features
df["job_category"] = df["job_title"].map(job_map)
df["location_tier"] = df["company_location"].map(tier_map)
df["is_same_country"] = (df["company_location"] == df["employee_residence"]).astype(int)

print(f"Dataset: {df.shape[0]} rows")
print(f"Job categories: {df['job_category'].unique()}")
print(f"Location tiers: {df['location_tier'].unique()}")
df[["job_category", "experience_level", "company_size", "location_tier", "remote_ratio", "is_same_country", "salary_in_usd"]].head()

## Step 2: Establish Baselines

Median `salary_in_usd` per `job_category` — this is the reference point for all deviation analysis.

In [ ]:
# Baseline: median salary per job category
baselines = df.groupby("job_category")["salary_in_usd"].median()
print("Per-category baseline medians:")
print(baselines.to_string())
print(f"\nOverall median: ${df['salary_in_usd'].median():,.0f}")

# Add baseline and deviation columns to the dataframe
df["baseline_median"] = df["job_category"].map(baselines)
df["pct_deviation"] = ((df["salary_in_usd"] - df["baseline_median"]) / df["baseline_median"]) * 100

# Quick sanity check
print(f"\nDeviation stats:")
print(df["pct_deviation"].describe().to_string())

## Step 3: Analyze Deviations

### 3.1 Experience Level

In [ ]:
exp_dev = df.groupby("experience_level")["pct_deviation"].agg(["mean", "median", "count"])
exp_dev.columns = ["mean_dev_%", "median_dev_%", "n"]
exp_dev = exp_dev.reindex(["EN", "MI", "SE", "EX"])
print("Experience Level — % Deviation from Category Baseline:")
print(exp_dev.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#d9534f" if v < 0 else "#5cb85c" for v in exp_dev["mean_dev_%"]]
exp_dev["mean_dev_%"].plot(kind="barh", ax=ax, color=colors)
ax.set_xlabel("Mean % Deviation from Baseline")
ax.set_title("Salary Deviation by Experience Level")
ax.axvline(0, color="black", linewidth=0.8)
for i, v in enumerate(exp_dev["mean_dev_%"]):
    ax.text(v + (2 if v >= 0 else -2), i, f"{v:+.1f}%", va="center", fontsize=10)
plt.tight_layout()
plt.show()

### 3.2 Location Tier

In [ ]:
tier_dev = df.groupby("location_tier")["pct_deviation"].agg(["mean", "median", "count"])
tier_dev.columns = ["mean_dev_%", "median_dev_%", "n"]
tier_dev = tier_dev.reindex(["High_Tier", "Mid_Tier", "Low_Tier"])
print("Location Tier — % Deviation from Category Baseline:")
print(tier_dev.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#d9534f" if v < 0 else "#5cb85c" for v in tier_dev["mean_dev_%"]]
tier_dev["mean_dev_%"].plot(kind="barh", ax=ax, color=colors)
ax.set_xlabel("Mean % Deviation from Baseline")
ax.set_title("Salary Deviation by Location Tier")
ax.axvline(0, color="black", linewidth=0.8)
for i, v in enumerate(tier_dev["mean_dev_%"]):
    ax.text(v + (2 if v >= 0 else -2), i, f"{v:+.1f}%", va="center", fontsize=10)
plt.tight_layout()
plt.show()

### 3.3 Company Size Penalty (SE/EX at Small vs Large)

In [ ]:
# Company size deviation for all experience levels
size_exp = df.groupby(["experience_level", "company_size"])["pct_deviation"].mean().unstack()
size_exp = size_exp.reindex(index=["EN", "MI", "SE", "EX"], columns=["S", "M", "L"])
print("Mean % Deviation by Experience Level x Company Size:")
print(size_exp.round(1).to_string())

# Focus: SE/EX penalty at Small vs Large
print("\n--- Senior/Executive Size Penalty ---")
for exp in ["SE", "EX"]:
    senior = df[df["experience_level"] == exp]
    small_med = senior[senior["company_size"] == "S"]["salary_in_usd"].median()
    large_med = senior[senior["company_size"] == "L"]["salary_in_usd"].median()
    n_s = len(senior[senior["company_size"] == "S"])
    n_l = len(senior[senior["company_size"] == "L"])
    if large_med > 0:
        penalty = ((small_med - large_med) / large_med) * 100
        print(f"{exp}: Small=${small_med:,.0f} (n={n_s}) vs Large=${large_med:,.0f} (n={n_l}) → {penalty:+.1f}% gap")

fig, ax = plt.subplots(figsize=(8, 4))
size_exp.plot(kind="bar", ax=ax)
ax.set_ylabel("Mean % Deviation from Baseline")
ax.set_title("Salary Deviation: Experience Level x Company Size")
ax.axhline(0, color="black", linewidth=0.8)
ax.legend(title="Company Size")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 3.4 International Remote (is_same_country)

In [ ]:
intl_dev = df.groupby("is_same_country")["pct_deviation"].agg(["mean", "median", "count"])
intl_dev.columns = ["mean_dev_%", "median_dev_%", "n"]
intl_dev.index = intl_dev.index.map({0: "International (diff country)", 1: "Same country"})
print("International vs Same Country — % Deviation:")
print(intl_dev.to_string())

# Salary comparison
print("\n--- Median Salary ---")
for val, label in [(1, "Same country"), (0, "International")]:
    subset = df[df["is_same_country"] == val]
    print(f"  {label}: ${subset['salary_in_usd'].median():,.0f} (n={len(subset)})")

fig, ax = plt.subplots(figsize=(6, 3))
colors = ["#d9534f" if v < 0 else "#5cb85c" for v in intl_dev["mean_dev_%"]]
intl_dev["mean_dev_%"].plot(kind="barh", ax=ax, color=colors)
ax.set_xlabel("Mean % Deviation from Baseline")
ax.set_title("Salary Deviation: International vs Same Country")
ax.axvline(0, color="black", linewidth=0.8)
for i, v in enumerate(intl_dev["mean_dev_%"]):
    ax.text(v + (2 if v >= 0 else -2), i, f"{v:+.1f}%", va="center", fontsize=10)
plt.tight_layout()
plt.show()

### 3.5 Remote Ratio

In [ ]:
remote_dev = df.groupby("remote_ratio")["pct_deviation"].agg(["mean", "median", "count"])
remote_dev.columns = ["mean_dev_%", "median_dev_%", "n"]
print("Remote Ratio — % Deviation from Category Baseline:")
print(remote_dev.to_string())

fig, ax = plt.subplots(figsize=(6, 3))
colors = ["#d9534f" if v < 0 else "#5cb85c" for v in remote_dev["mean_dev_%"]]
remote_dev["mean_dev_%"].plot(kind="barh", ax=ax, color=colors)
ax.set_xlabel("Mean % Deviation from Baseline")
ax.set_title("Salary Deviation by Remote Ratio")
ax.axvline(0, color="black", linewidth=0.8)
for i, v in enumerate(remote_dev["mean_dev_%"]):
    ax.text(v + (2 if v >= 0 else -2), i, f"{v:+.1f}%", va="center", fontsize=10)
plt.tight_layout()
plt.show()

### 3.6 Summary: Deviation Magnitudes (All Features Side-by-Side)

In [ ]:
# Combine all deviations for ranking
summary = pd.DataFrame({
    "feature": [
        "EX (Executive)", "SE (Senior)", "MI (Mid)", "EN (Entry)",
        "High_Tier", "Mid_Tier", "Low_Tier",
        "Same country", "International",
        "Remote 100%", "Remote 50%", "Remote 0%",
    ],
    "mean_dev_%": [
        exp_dev.loc["EX", "mean_dev_%"], exp_dev.loc["SE", "mean_dev_%"],
        exp_dev.loc["MI", "mean_dev_%"], exp_dev.loc["EN", "mean_dev_%"],
        tier_dev.loc["High_Tier", "mean_dev_%"], tier_dev.loc["Mid_Tier", "mean_dev_%"],
        tier_dev.loc["Low_Tier", "mean_dev_%"],
        intl_dev.loc["Same country", "mean_dev_%"],
        intl_dev.loc["International (diff country)", "mean_dev_%"],
        remote_dev.loc[100, "mean_dev_%"], remote_dev.loc[50, "mean_dev_%"],
        remote_dev.loc[0, "mean_dev_%"],
    ]
})

summary = summary.sort_values("mean_dev_%", ascending=True)
print("All Feature Deviations Ranked (strongest negative → strongest positive):")
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#d9534f" if v < 0 else "#5cb85c" for v in summary["mean_dev_%"]]
ax.barh(summary["feature"], summary["mean_dev_%"], color=colors)
ax.set_xlabel("Mean % Deviation from Category Baseline")
ax.set_title("Feature Attribution: Salary Deviation Drivers")
ax.axvline(0, color="black", linewidth=0.8)
for i, (_, row) in enumerate(summary.iterrows()):
    v = row["mean_dev_%"]
    ax.text(v + (1 if v >= 0 else -1), i, f"{v:+.1f}%", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## Step 4: Extract Python Logic — `determine_primary_driver`

**Findings ranked by absolute deviation magnitude:**

| Driver | Mean Deviation | Direction |
|--------|---------------|-----------|
| EX (Executive) | +83.7% | Strongest positive |
| Low_Tier geography | -75.5% | Strongest negative |
| Mid_Tier geography | -46.4% | Strong negative |
| EN (Entry-level) | -36.2% | Strong negative |
| SE (Senior) | +35.1% | Positive |
| High_Tier geography | +26.4% | Positive |
| EX at Small company | -40.0% vs Large | Interaction penalty |
| SE at Small company | -26.1% vs Large | Interaction penalty |
| International remote | -23.3% | Moderate negative |
| Remote 100% | +19.3% | Moderate positive |

**Priority logic tree:**
1. Entry-level (EN) below → experience is the primary drag
2. Executive (EX) above → seniority premium
3. Low/Mid tier below → geographic pay gap
4. High tier above → high-tier geography premium
5. Senior at Small below → company size penalty
6. International → cross-border discount
7. Fallback → general market positioning

In [ ]:
def determine_primary_driver(combo: dict, status: str) -> str:
    """
    Determine the primary driver of a salary prediction's deviation from
    the market baseline, based on EDA-derived feature attribution rules.

    Parameters
    ----------
    combo : dict
        The input combination dict with keys: experience_level, location_tier,
        company_size, is_same_country, remote_ratio.
    status : str
        "ABOVE" or "BELOW" — the pre-calculated position vs market median.

    Returns
    -------
    str
        A phrase completing: "This positioning is primarily driven by [X]."
    """
    exp = combo.get("experience_level", "")
    tier = combo.get("location_tier", "")
    size = combo.get("company_size", "")
    same = combo.get("is_same_country", 1)
    remote = combo.get("remote_ratio", 50)

    # ── Priority 1: Entry-level is the strongest negative drag (-36%) ─────
    if exp == "EN" and status == "BELOW":
        return "entry-level experience, which averages 36% below category baselines"

    # ── Priority 2: Executive seniority premium (+84%) ────────────────────
    if exp == "EX" and status == "ABOVE":
        return "executive-level seniority, which commands an 84% premium over category baselines"

    # ── Priority 3: Low/Mid-tier geography penalty (-75% / -46%) ──────────
    if tier == "Low_Tier" and status == "BELOW":
        return "the low-tier geographic market, where salaries average 75% below global baselines"

    if tier == "Mid_Tier" and status == "BELOW":
        return "the mid-tier geographic market, where salaries average 46% below global baselines"

    # ── Priority 4: High-tier geography premium (+26%) ────────────────────
    if tier == "High_Tier" and status == "ABOVE":
        return "the high-tier geographic market, which sustains a 26% premium over global baselines"

    # ── Priority 5: Senior/Executive at Small company penalty ─────────────
    if exp in ("SE", "EX") and size == "S" and status == "BELOW":
        pct = "40%" if exp == "EX" else "26%"
        return f"the small-company size penalty, where {exp}-level roles earn {pct} less than at large firms"

    # ── Priority 6: International cross-border discount (-23%) ────────────
    if same == 0 and status == "BELOW":
        return "the international cross-border arrangement, which averages 23% below domestic roles"

    # ── Priority 7: Fallback — general market positioning ─────────────────
    if status == "ABOVE":
        return "a combination of seniority, geography, and company scale favoring above-market compensation"
    return "a combination of experience level, geographic market, and company characteristics"


# ── Test the function ─────────────────────────────────────────────────────────
test_cases = [
    ({"experience_level": "EN", "location_tier": "High_Tier", "company_size": "S", "is_same_country": 1, "remote_ratio": 0}, "BELOW"),
    ({"experience_level": "EX", "location_tier": "High_Tier", "company_size": "L", "is_same_country": 1, "remote_ratio": 100}, "ABOVE"),
    ({"experience_level": "SE", "location_tier": "Low_Tier", "company_size": "M", "is_same_country": 1, "remote_ratio": 50}, "BELOW"),
    ({"experience_level": "MI", "location_tier": "High_Tier", "company_size": "L", "is_same_country": 0, "remote_ratio": 100}, "BELOW"),
    ({"experience_level": "SE", "location_tier": "High_Tier", "company_size": "S", "is_same_country": 1, "remote_ratio": 50}, "BELOW"),
    ({"experience_level": "SE", "location_tier": "High_Tier", "company_size": "L", "is_same_country": 1, "remote_ratio": 100}, "ABOVE"),
]

print("Test cases:")
for combo, status in test_cases:
    driver = determine_primary_driver(combo, status)
    label = f"{combo['experience_level']} | {combo['location_tier']} | {combo['company_size']} | same={combo['is_same_country']}"
    print(f"  [{status:5s}] {label}")
    print(f"         → This positioning is primarily driven by {driver}.")
    print()